In [1]:
import os, yaml, sys
import numpy as np
import torch
import h5py
from torchvision import models
from scipy.spatial.distance import euclidean,squareform, pdist
import matplotlib.pyplot as plt
from scipy.io import loadmat
from numba import njit
ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
from general_utils.utils import get_device, print_wise, index_gram
device = get_device()

15:44:26 - device being used: mps


In [2]:
def cosine_sim(x):
    norm = np.sqrt(np.sum(x**2, axis=0))
    x = x/norm
    gram = x.T @ x 
    gram = 1- gram
    gram = index_gram(gram)
    return gram
# EOF


In [3]:
@njit
def euclidean_dist_sing(v1, v2):
    dist_sq = 0
    for i in range(v1.shape[0]):
        diff = v1[i]-v2[i]
        diff_sq = diff**2
        dist_sq+= diff_sq
    dist = np.sqrt(dist_sq)
    return dist
# EOF

In [4]:
@njit
def euclidean_dist_mat(x):
    n = x.shape[1]
    n_pairs = n * (n - 1) // 2
    gram_vec = np.empty(n_pairs)
    counter = 0
    for i in range(n):
        v1 = x[:,i]
        for j in range(i+1, n):
            v2 = x[:,j]
            dist = euclidean_dist_sing(v1, v2)
            gram_vec[counter] = dist
            counter += 1
    return gram_vec
# EOF


In [5]:
v1 = np.random.randn(int(10e8))
v2 = np.random.randn(int(10e8))
%timeit -n 1 -r 1 euclidean_dist_sing(v1, v2)
%timeit -n 1 -r 1 np.linalg.norm(v1 - v2)
%timeit -n 1 -r 1 euclidean(v1, v2)

35.8 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
1min 46s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
1min 38s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [9]:
v1 = np.random.randn(int(10e3),int(10e2))
%timeit -n 1 -r 1 euclidean_dist_mat(v1)
%timeit -n 1 -r 1 pdist(v1.T, metric="euclidean") 
%timeit -n 1 -r 1 pdist(v1.T, metric=euclidean_dist_sing) 

11.7 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
5.41 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
11.4 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


True